# Hierarchical Risk Parity

Lopez de Prado (2016) builds a tree of the assets from their correlations, reorders the covariance matrix along it, then splits the weight between halves of the ordered list in inverse proportion to their risk. No covariance matrix is inverted and no expected returns are used. This notebook looks at the correlations, computes HRP weights and compares them with inverse volatility.

In [ ]:
Pollis.packages("PortfolioOptimisers", "Clarabel", "Plots")

In [ ]:
using PortfolioOptimisers, Clarabel, Random, Statistics, Plots

## The data

1,000 trading days of daily simple returns for 8 assets in two sectors. A market factor moves every asset, a sector factor moves each sector, and Student-t shocks with 5 degrees of freedom give fat tails. The true annualised means are 12% for Tech and 6% for Util, so the estimates can be checked against them.

In [ ]:
Random.seed!(42)
T, N = 1_000, 8
t5(n) = sqrt(3 / 5) .* randn(n) ./ sqrt.(vec(sum(abs2, randn(5, n); dims = 1)) ./ 5)   # unit-variance Student-t(5) shocks
mkt = 0.008 .* t5(T); tech = 0.010 .* t5(T); util = 0.005 .* t5(T)   # market and sector factors
mu_d = [fill(0.12, 4); fill(0.06, 4)] ./ 252   # true daily means
R = hcat([j <= 4 ? mu_d[j] .+ 1.2 .* mkt .+ tech .+ 0.012 .* t5(T) : mu_d[j] .+ 0.6 .* mkt .+ util .+ 0.006 .* t5(T) for j in 1:N]...)
asset_names = ["Tech1", "Tech2", "Tech3", "Tech4", "Util1", "Util2", "Util3", "Util4"]

rd = ReturnsResult(; nx = asset_names, X = R)
rf_daily = 0.02 / 252
slv = Solver(; name = :clarabel, solver = Clarabel.Optimizer, settings = Dict("verbose" => false),
             check_sol = (; allow_local = true, allow_almost = true))
show_weights(w) = foreach((nm, wi) -> println(rpad(nm, 8), lpad(round(100 * wi; digits = 2), 8), " %"), asset_names, w)

## The correlations

Blocks along the diagonal are the clusters the tree will find.

In [ ]:
heatmap(asset_names, asset_names, cor(R); clims = (-1, 1), color = :RdBu, yflip = true, xrotation = 45)

## HRP weights

In [ ]:
res = optimise(HierarchicalRiskParity(; r = Variance()), rd)
w_hrp, pr = res.w, res.pr
show_weights(w_hrp)
println("Tech share: ", round(100 * sum(w_hrp[1:4]); digits = 1), " %   Util share: ", round(100 * sum(w_hrp[5:8]); digits = 1), " %")

## Share of risk

Each asset's Euler contribution to variance, against its share of capital.

In [ ]:
rc = risk_contribution(Variance(), w_hrp, pr)
bar(asset_names, 100 .* rc ./ sum(rc); label = "Share of Risk", ylabel = "%", xrotation = 45)
scatter!(asset_names, 100 .* w_hrp; label = "Share of Capital")

## Against inverse volatility

Inverse volatility weights each asset by $1/\sigma_i$ and ignores correlations. The difference shows what the tree adds.

In [ ]:
w_iv = optimise(InverseVolatility(), rd).w
bar(asset_names, 100 .* w_hrp; label = "HRP", ylabel = "Weight (%)", xrotation = 45)
scatter!(asset_names, 100 .* w_iv; label = "Inverse Volatility", ms = 6)

## Exercises

1. Set the sector factors `tech` and `util` to zero, so that only the market factor links the assets. How do the heatmap and the HRP weights change?
2. Add a ninth asset that copies `Tech1` plus a little noise. How does HRP share the weight between the two?
3. Compute the minimum-variance weights on the same data and compare their effective number of assets, `1 / sum(abs2, w)`, with HRP's.